# 10. Construcción del catálogo de artículos

Reunir en una fila por artículo sus datos bibliográficos y los nombres de sus autores UNAM registrados en la base 2024–2025. Se usa `indice` como identificador porque hay artículos con el mismo título.

## 1. Leer la base final

In [5]:
import os
import pandas as pd

fuente = "04_Limpieza/06_Finales/Canonico_FINAL.csv"
archivo = "../" + fuente
carpeta_salida = "../06_grafos"

if not os.path.exists(archivo):
    archivo = fuente
    carpeta_salida = "06_grafos"

base = pd.read_csv(archivo, dtype=str, keep_default_na=False, encoding="utf-8-sig")
print(f"Registros artículo-autor: {len(base):,}")

Registros artículo-autor: 1,151


## 2. Comprobar los datos

Se revisa que no haya autores vacíos, que `indice` y `Año` sean enteros y que los metadatos de un mismo artículo coincidan en todas sus filas.

In [6]:
metadatos = ["Titulo", "Año", "Area", "Keywords", "Abstract", "Doi", "URL"]

print("Filas sin autor:", (base["Autor_norm"].str.strip() == "").sum())
print("Índices no numéricos:", (~base["indice"].str.fullmatch(r"[0-9]+")).sum())
print("Años no numéricos:", (~base["Año"].str.fullmatch(r"[0-9]+")).sum())

datos = base.copy()
datos["indice"] = datos["indice"].astype(int)
datos["Año"] = datos["Año"].astype(int)

# Revisar si algún artículo tiene valores distintos en sus metadatos
diferencias = datos.groupby("indice")[metadatos].nunique() > 1
print("Artículos con diferencias en sus metadatos:", diferencias.any(axis=1).sum())

Filas sin autor: 0
Índices no numéricos: 0
Años no numéricos: 0
Artículos con diferencias en sus metadatos: 0


## 3. Reunir los autores de cada artículo

`article_id` conserva el valor de `indice`. Los nombres de los autores se separan con punto y coma en el orden en que aparecen en la base.

In [7]:
filas = []

for indice, grupo in datos.groupby("indice", sort=True):
    primera = grupo.iloc[0]
    autores_unicos = list(dict.fromkeys(grupo["Autor_norm"]))
    
    filas.append({
        "article_id": indice,
        "titulo": primera["Titulo"],
        "autores": "; ".join(autores_unicos),
        "anio": primera["Año"],
        "area": primera["Area"],
        "keywords": primera["Keywords"],
        "abstract": primera["Abstract"],
        "doi": primera["Doi"],
        "url": primera["URL"]
    })

catalogo = pd.DataFrame(filas)

print(f"Catálogo: {len(catalogo)} artículos y {len(catalogo.columns)} columnas.")
print(f"Nombres de autor distintos: {base['Autor_norm'].nunique()}")
catalogo[["article_id", "titulo", "autores", "anio", "area"]].head()

Catálogo: 519 artículos y 9 columnas.
Nombres de autor distintos: 628


,article_id,titulo,autores,anio,area
0,1,A comparative study of NeuralODE and Universal...,Raymundo Vazquez Martinez,2024,CC
1,2,A Comparative Study of Numerical Laplace Inver...,Vladimir Martínez Bernardino,2025,CC
2,3,A Greenfield Analysis for Supply Chains Enhanc...,Jose Antonio Marmolejo Saucedo,2024,CC
3,4,A Hierarchical Approach to a Tri-Objective Por...,Carlos Ignacio Hernández Castellanos; Yeudiel ...,2024,CC
4,5,A hybrid framework for assessing regional iner...,Mario Roberto Arrieta Paternina,2025,CC


## 4. Guardar el catálogo

Se guardan `catalogo_articulos.xlsx` (hoja `Articulos`) y `catalogo_articulos.csv` en `06_grafos/`.

In [8]:
os.makedirs(carpeta_salida, exist_ok=True)

catalogo.to_excel(
    carpeta_salida + "/catalogo_articulos.xlsx",
    sheet_name="Articulos",
    index=False,
    engine="openpyxl"
)
catalogo.to_csv(
    carpeta_salida + "/catalogo_articulos.csv",
    index=False,
    encoding="utf-8-sig",
    lineterminator="\r\n"
)

print("Archivos guardados en 06_grafos/: catalogo_articulos.csv y catalogo_articulos.xlsx")

Archivos guardados en 06_grafos/: catalogo_articulos.csv y catalogo_articulos.xlsx
